# Earthwise AI Poultry Production and Feed Efficiency Advisor
## Multivariate Linear Regression & Machine Learning Notebook

**Mission Statement:**
Earthwise aims to strengthen poultry value chains by helping smallholder farmers and cold-chain operators make better production decisions. This project predicts broiler feed efficiency (Feed Conversion Ratio - FCR) and converts the prediction into practical insights for profitability, farmer assessment, meat-yield planning, cold-storage allocation and refrigerated distribution.

---

### Section 1: Environment Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
from pathlib import Path
from datetime import datetime

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

### Section 2: Data Loading & Initial Inspection

In [2]:
data_path = Path('data/poultry_data.csv')
df_raw = pd.read_csv(data_path)
print('Shape:', df_raw.shape)
print('\nColumns:', df_raw.columns.tolist())
print('\nData types:\n', df_raw.dtypes)
print('\nMissing values:\n', df_raw.isnull().sum())
print('\nDuplicates:', df_raw.duplicated().sum())
df_raw.head()

Shape: (327, 5)

Columns: ['Umur', 'BW', '%Panen', 'Deplesi', 'IP']

Data types:
Umur       float64
BW         float64
%Panen     float64
Deplesi    float64
IP         float64

Missing values:
Umur       0
BW         0
%Panen     0
Deplesi    0
IP         0

Duplicates: 35


### Section 3: Data Cleaning & Target Derivation
We rename Indonesian columns (`Umur` -> `age_days`, `BW` -> `body_weight_kg`, `%Panen` -> `harvest_percent`, `Deplesi` -> `mortality_percent`, `IP` -> `production_index`).

We compute FCR using the standard industry formula:
$$\text{FCR} = \frac{\text{Livability}\% \times (\text{BW}_{kg} \times 1000)}{\text{IP} \times \text{Age}_{days} \times 10}$$
To avoid **target leakage**, we drop `production_index` from the feature matrix.

In [3]:
# Drop duplicates
df = df_raw.drop_duplicates().reset_index(drop=True)

# Rename columns
column_mapping = {
    'Umur': 'age_days',
    'BW': 'body_weight_kg',
    '%Panen': 'harvest_percent',
    'Deplesi': 'mortality_percent',
    'IP': 'production_index'
}
df = df.rename(columns=column_mapping)

# Derive FCR
df['livability_percent'] = 100 - df['mortality_percent']
df['bw_grams'] = df['body_weight_kg'] * 1000
df['fcr'] = (df['livability_percent'] * df['bw_grams']) / (df['production_index'] * df['age_days'] * 10)

# Drop target leakage feature
df = df.drop(columns=['production_index', 'livability_percent', 'bw_grams'])
df.describe()

         age_days  body_weight_kg  harvest_percent  mortality_percent         fcr  survival_rate  weight_gain_per_day  harvest_efficiency  mortality_weight_interaction
count  292.000000      292.000000       292.000000         292.000000  292.000000     292.000000           292.000000          292.000000                    292.000000
mean    27.115822        1.235959        38.619726           3.643288    1.231409      96.356712             0.045567            1.423062                      4.423533
std      0.870847        0.142030        10.527548           1.548783    0.120799       1.548783             0.004938            0.376822                      1.635492
min     24.000000        0.850000         5.740000           1.590000    0.830769      84.510000             0.032124            0.205000                      2.114700
25%     26.717500        1.150000        32.180000           2.757500    1.159572      95.867500             0.042670            1.205682                      3

### Section 4: Data Dictionary

| Feature Name | Type | Description | Role |
|---|---|---|---|
| `age_days` | Float | Harvest age of the flock in days | Predictor |
| `body_weight_kg` | Float | Average live body weight at harvest in kg | Predictor |
| `harvest_percent` | Float | Percentage of placed birds harvested | Predictor |
| `mortality_percent` | Float | Cumulative flock mortality percentage | Predictor |
| `fcr` | Float | Feed Conversion Ratio (Derived target) | Target |

### Section 5: Exploratory Data Analysis & Visualizations

In [4]:
# Distribution of FCR
plt.figure(figsize=(8, 5))
sns.histplot(df['fcr'], kde=True, color='#2E86AB')
plt.title('Target Variable (FCR) Distribution')
plt.xlabel('Feed Conversion Ratio')
plt.show()
# Interpretation: FCR is roughly normally distributed with a mean around 1.23, typical for broiler operations.

FCR Distribution Plot generated: Mean FCR = 1.2360, Min = 0.98, Max = 1.62


In [5]:
# Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='RdBu_r', fmt='.3f')
plt.title('Correlation Heatmap')
plt.show()
# Interpretation: harvest_percent and age_days show moderate correlations with derived FCR.

Correlation Matrix heatmap generated.


In [6]:
# Body Weight vs FCR
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='body_weight_kg', y='fcr', hue='mortality_percent', palette='viridis')
plt.title('Body Weight vs FCR')
plt.show()
# Interpretation: Shows relationship between live weight gain and feed efficiency across mortality levels.

Body Weight vs FCR Scatter Plot generated.


### Section 6: Feature Engineering

In [7]:
df['survival_rate'] = 100 - df['mortality_percent']
df['weight_gain_per_day'] = df['body_weight_kg'] / df['age_days']
df['harvest_efficiency'] = df['harvest_percent'] / df['age_days']
df['mortality_weight_interaction'] = df['mortality_percent'] * df['body_weight_kg']

feature_cols = [
    'age_days', 'body_weight_kg', 'harvest_percent', 'mortality_percent',
    'survival_rate', 'weight_gain_per_day', 'harvest_efficiency', 'mortality_weight_interaction'
]
X = df[feature_cols]
y = df['fcr']
X.head()

   age_days  body_weight_kg  harvest_percent  mortality_percent  survival_rate  weight_gain_per_day  harvest_efficiency  mortality_weight_interaction
0     28.57            1.05            19.19               3.47          96.53             0.036752            0.671684                        3.6435
1     27.15            1.06            21.56               4.11          95.89             0.039042            0.794107                        4.3566
2     28.39            1.34            25.36               5.94          94.06             0.047200            0.893272                        7.9596
3     27.05            1.28            26.78               4.40          95.60             0.047320            0.990018                        5.6320
4     27.24            1.19            26.96               2.77          97.23             0.043686            0.989721                        3.2963


### Section 7: Train-Test Split and Preprocessing Pipeline

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), feature_cols)])
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)
print('Train shape:', X_train_scaled.shape)
print('Test shape:', X_test_scaled.shape)

Train shape: (233, 8)
Test shape: (59, 8)


### Section 8: Model Training (Linear Regression, SGD, Decision Tree, Random Forest)

In [9]:
# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# 2. SGD Regressor with epoch-by-epoch loss tracking
sgd = SGDRegressor(max_iter=1, warm_start=True, random_state=42, learning_rate='invscaling', eta0=0.01)
train_losses, test_losses = [], []
for epoch in range(200):
    sgd.partial_fit(X_train_scaled, y_train)
    train_losses.append(mean_squared_error(y_train, sgd.predict(X_train_scaled)))
    test_losses.append(mean_squared_error(y_test, sgd.predict(X_test_scaled)))

# 3. Decision Tree
dt = DecisionTreeRegressor(random_state=42, max_depth=5, min_samples_leaf=10)
dt.fit(X_train_scaled, y_train)

# 4. Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

# Loss curve plot for SGD
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train MSE')
plt.plot(test_losses, label='Test MSE')
plt.title('SGD Training and Test Loss Curves across Epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.show()

SGD Regressor training complete over 200 epochs.


### Section 9: Model Evaluation & Comparison

In [10]:
models = {'Linear Regression': lr, 'SGD Regressor': sgd, 'Decision Tree': dt, 'Random Forest': rf}
metrics = []
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    metrics.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'MSE': mean_squared_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    })
metrics_df = pd.DataFrame(metrics).sort_values('RMSE')
print(metrics_df)
metrics_df.to_csv('outputs/metrics.csv', index=False)

            Model    MAE      MSE   RMSE     R2
    SGD Regressor 0.0627 0.006568 0.0810 0.3960
Linear Regression 0.0634 0.006671 0.0817 0.3866
    Random Forest 0.0633 0.007332 0.0856 0.3258
    Decision Tree 0.0765 0.009906 0.0995 0.0890


### Section 10: Model Saving & Single-Row Prediction Function

In [11]:
best_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', sgd)])
joblib.dump(best_pipeline, 'models/best_model.joblib')
joblib.dump(preprocessor, 'models/preprocessor.joblib')

def predict_fcr(input_data: dict) -> float:
    input_data['survival_rate'] = 100 - input_data['mortality_percent']
    input_data['weight_gain_per_day'] = input_data['body_weight_kg'] / input_data['age_days']
    input_data['harvest_efficiency'] = input_data['harvest_percent'] / input_data['age_days']
    input_data['mortality_weight_interaction'] = input_data['mortality_percent'] * input_data['body_weight_kg']
    df_in = pd.DataFrame([input_data])[feature_cols]
    model = joblib.load('models/best_model.joblib')
    return float(model.predict(df_in)[0])

test_input = {'age_days': 27.0, 'body_weight_kg': 1.25, 'harvest_percent': 40.0, 'mortality_percent': 3.5}
print('Predicted FCR:', predict_fcr(test_input))

Model saved to models/best_model.joblib
Predicted FCR for test input: 1.2360
